# 🚀 F5-TTS Sorunsuz Türkçe Ses Klonlama
Eski motorların (Coqui/Chatterbox) Colab'daki bağımlılık sorunlarını aşmak için, en modern ve sorunsuz çalışan Flow Matching mimarili **F5-TTS** modelini kullanıyoruz.

⚠️ **ÖNEMLİ:** Menüden `Runtime -> Disconnect and delete runtime` (Çalışma zamanını kes ve sil) diyerek tamamen sıfırdan, temiz bir GPU (T4) ile başlayın.

## Hücre 1: Google Drive Bağlantısı

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/BITIRME"

ref_dir = os.path.join(PROJECT_DIR, "data/reference_voice")
if os.path.exists(ref_dir):
    print(f"✅ {len(os.listdir(ref_dir))} referans ses bulundu.")
else:
    print(f"❌ Referans klasörü bulunamadı: {ref_dir}")

## Hücre 2: F5-TTS Temiz Kurulum
Hata vermeden kurulur ve çalışır. Kurulum bitince **Yeniden Başlatmaya GEREK YOKTUR.**

In [ ]:
!pip install -q git+https://github.com/SWivid/F5-TTS.git
!pip install -q soundfile scipy numpy

print("✅ F5-TTS Kurulumu Tamamlandı!")

## Hücre 3: Test Cümleleri ve Hazırlık

In [ ]:
import os, time
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from scipy.signal import butter, sosfilt

TEST_SENTENCES = {
    "neutral": "Merhaba, ben bankadan arıyorum. Hesabınızla ilgili bir bilgilendirme yapmak istiyorum.",
    "urgent": "Dikkat! Hesabınızda şüpheli bir işlem tespit ettik. Hemen doğrulama yapmanız gerekiyor.",
    "friendly": "İyi günler, size özel bir kampanyamız var. Birkaç dakikanızı alabilir miyim?",
    "worried": "Bir dakika, bu biraz garip geldi. Gerçekten bankadan mı arıyorsunuz?",
    "threatening": "Eğer beş dakika içinde bilgilerinizi doğrulamazsanız, hesabınız kalıcı olarak kapatılacak.",
}

AGENT_REF = os.path.join(PROJECT_DIR, "data/reference_voice/agent.wav")
VICTIM_REF = os.path.join(PROJECT_DIR, "data/reference_voice/victim.wav")

OUTPUT_DIR = "/content/f5_tts_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Post-Processing Fonksiyonu
def postprocess(audio, sr):
    sos = butter(4, 80.0, btype='high', fs=sr, output='sos')
    audio = sosfilt(sos, audio).astype(np.float32)
    sos = butter(2, [200, 400], btype='band', fs=sr, output='sos')
    audio = (audio + sosfilt(sos, audio) * 0.33).astype(np.float32)
    sos = butter(2, [3000, 5000], btype='band', fs=sr, output='sos')
    audio = (audio - sosfilt(sos, audio) * 0.29).astype(np.float32)
    threshold = 10.0 ** (-18.0 / 20.0)
    out = audio.copy()
    mask = np.abs(out) > threshold
    above = np.abs(out[mask])
    out[mask] = np.sign(out[mask]) * (threshold + (above - threshold) / 3.0)
    audio = out.astype(np.float32)
    peak = np.max(np.abs(audio))
    if peak > 1e-8:
        audio = (audio * (0.89 / peak)).astype(np.float32)
    return audio

print("✅ Hazırlık tamam.")

## Hücre 4: F5-TTS Sentez

In [ ]:
import torch
from f5_tts.infer.utils_infer import load_model, load_vocoder, infer_process

device = "cuda" if torch.cuda.is_available() else "cpu"
print("F5-TTS modeli indiriliyor ve yükleniyor (Birkaç dakika sürebilir)...")

model_obj = load_model(
    model_cls=None, model_cfg=None, ckpt_path=None,
    mel_spec_type="vocos", vocab_file=None, ode_method="euler",
    use_ema=True, device=device
)
vocoder = load_vocoder(vocoder_name="vocos", is_local=False, local_path="", device=device)

print(f"✅ F5-TTS başarıyla yüklendi ({device})")
print("="*50)

results = {}
for emotion, text in TEST_SENTENCES.items():
    ref = VICTIM_REF if emotion == "worried" else AGENT_REF
    out = os.path.join(OUTPUT_DIR, f"f5_{emotion}.wav")
    pp  = os.path.join(OUTPUT_DIR, f"f5_{emotion}_pp.wav")

    print(f"\n🎭 [{emotion}] {text[:50]}...")
    start = time.time()
    
    # Sentez
    audio, sr, _ = infer_process(
        ref_audio=ref,
        ref_text="", # Boş bırakılırsa model referans sesi otomatik analiz eder
        gen_text=text,
        model_obj=model_obj,
        vocoder=vocoder,
        mel_spec_type="vocos",
        speed=1.0
    )
    elapsed = time.time() - start
    
    # Kaydet ve PP uygula
    sf.write(out, audio, sr)
    pp_audio = postprocess(audio, sr)
    sf.write(pp, pp_audio, sr)
    
    print(f"   ✅ {len(audio)/sr:.1f}s ses — {elapsed:.1f}s sürdü")
    print("   🔊 Ham Sentez:")
    display(Audio(out))
    print("   🔊 Post-processed (Robotsu Ses Giderici):")
    display(Audio(pp))

print("\n✅ İşlem tamamlandı! Ses dosyaları Drive'a kopyalanıyor...")

# Drive'a Kopyala
import shutil
save_dir = os.path.join(PROJECT_DIR, "outputs/f5_tts_test")
os.makedirs(save_dir, exist_ok=True)
for fname in os.listdir(OUTPUT_DIR):
    shutil.copy2(os.path.join(OUTPUT_DIR, fname), os.path.join(save_dir, fname))
print(f"📁 Dosyalar şuraya kaydedildi: {save_dir}")